# Detector: which cells moved this week?

The detector runs every Monday on the review week **Monday to Sunday** (`AS_OF` is that Sunday). No LLM.
It scores every metric at every level through four lenses:

| Lens | Compares | Cancels out |
|---|---|---|
| **A** `A_yoy_shift` | this week vs last year, minus the same for the prior 4 weeks | seasonality, holidays that repeat |
| **B** `B_vs_peers` | this cell vs its peers (iOS vs other platforms, NYC vs other markets) | flu, holidays, anything that hits every peer |
| **D** | A and B on every funnel step, not only bookings | (catches offsetting moves inside the funnel) |
| **E** `E_consistency` | daily app bookings vs scheduler bookings | (should be equal) |

Your job in this notebook: pick the two numbers in the **flag rule**:
`abs(z) >= Z AND abs(bookings_impact) >= FLOOR`. The z gate asks "is the move real?", the impact floor asks
"is it worth acting on?" (in bookings per week, so revenue). Look at the scores (sections 1–2), pick the numbers
from the **alert budget** table (section 3, uses no answers), then check once against the dev set (section 4).

In [ ]:
import datetime as dt, importlib, json, subprocess, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import agents.detector as det
from agents.db import Database
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

AS_OF = dt.date(2025, 12, 28)          # a Sunday: holiday week, iOS tracking bug, noisy small cells. Try 2025-11-16 too
db = Database(AS_OF)
print("review week:", det.review_week(AS_OF), "to", AS_OF)

## 1. The weekly KPI tree

`bookings = searches × availability_rate × click_rate × booking_cvr`. `date_trunc('week', date)` starts weeks on Monday,
which is our review week. Change the market and specialty to look around.

In [ ]:
db.query("""
SELECT date_trunc('week', date)::DATE AS week,
       sum(searches) AS searches,
       round(sum(searches_with_availability) / sum(searches), 3) AS availability_rate,
       round(sum(profile_views) / sum(searches_with_availability), 3) AS click_rate,
       round(sum(bookings) / sum(profile_views), 3) AS booking_cvr,
       sum(bookings) AS bookings
FROM daily_funnel
WHERE market = 'Chicago' AND specialty = 'Dermatology'
GROUP BY 1 ORDER BY 1 DESC
LIMIT 8
""")

## 2. Scores for the review week

One row per cell × metric × lens, before any rule. `'all'` means summed over that dimension.

- `value`, `value_prev4`: this week, and the prior 4 weeks combined
- `value_compared`: last year's same week (lens A) or the peers this week (lens B)
- `r`: the residual, a log ratio. `r = -0.10` means about 10% below what was expected
- `sigma`: this cell's normal noise in `r` (robust spread over the previous 26 weeks); `z = r / sigma`
- `base_volume`: weekly volume behind the metric (searches for availability, profile views for CVR, the count itself for counts)
- `bookings_impact`: bookings per week this move adds or costs, `(e^r − 1) × the cell's average weekly bookings`

In [ ]:
s = det.scores(db)
cols = ["lens", "compared_with", "market", "specialty", "platform", "metric", "value", "value_prev4", "r", "z", "base_volume", "bookings_impact"]
print(len(s), "scored rows")
s.reindex(s.z.abs().sort_values(ascending=False).index)[cols].head(20).round(3)

Two kinds of rows have a big `z` but may not be worth an analyst's time. Look at both before choosing a rule:

- **big z, tiny move**: huge cells have so little noise that a 1% change gets a big z
- **big z, tiny volume**: small cells (e.g. flu shot, ~1–2 providers a day in some markets) jump around

In [ ]:
big_z = s[s.z.abs() >= 3]
print("|z| >= 3:", len(big_z), "rows")
print("\n...of which the move is under 3%:")
display(big_z[big_z.r.abs() < 0.03][cols].round(3).head(10))
print("...of which base_volume is under 100 a week:")
display(big_z[big_z.base_volume < 100][cols].round(3).head(10))

## 3. Pick the numbers from the alert budget (no answers used)

Quiet weeks (Jun 8 – Sep 28, 2025) have nothing planted, so every flag there is a false alarm. The table shows
flags per quiet week for each z cutoff (rows) and bookings-impact floor (columns). Pick the loosest rule that fits
the budget (about 5 a week). Looser = catches more, costs more agent tokens and reading time.
For scale: Phoenix books ~1,500 a week, NYC ~4,000; all markets ~12,400.

The first run takes ~20 s (scores are cached).

In [ ]:
from evals.alert_budget import flags_per_quiet_week
flags_per_quiet_week()

In [ ]:
Z, FLOOR = 4.0, 20                     # your pick
RULE = f"abs(z) >= {Z} AND abs(bookings_impact) >= {FLOOR}"
RULE

## 4. Dev-set sanity check (look once, after choosing)

This reads the answer key. Use it to confirm nothing is badly broken, not to re-tune: changing the numbers
because of this table is overfitting to the 4 planted events.

In [ ]:
from evals.detector_check import check
out = check(RULE)
print(f"scenarios right: {out['scenarios_right']}")
out["table"]

## 5. What the agent receives

The final output for one week: E flags first (data problems change how every other number reads), then the top 15 by |bookings_impact|.

In [ ]:
importlib.reload(det)
print(json.dumps(det.detect(AS_OF, rule=RULE), indent=1)[:3000])

## 6. Save your rule

Put your two numbers into `FLAG_RULE` in `agents/detector.py`, then run the tests.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-1500:])